In [1]:
import pandas as pd
import duckdb as ddb
from pathlib import Path

In [2]:
DATAPATH = '/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/full_nodes_featurestwo.parquet'

In [3]:
con = ddb.connect(':memory:')

In [4]:
# importing the raw node data
# read parquet
con.execute(f"""CREATE TABLE nodes_table AS SELECT * FROM read_parquet('{DATAPATH}')""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
con.query("""SELECT * FROM nodes_table LIMIT 5""").show()

┌───────────────────┬──────────────────┬──────────────────┬────────────────────┬──────────────────┬──────────────────┬───────────────────────┬────────────────────────┬──────────────────────┬───────────────────────┬─────────────────────┬─────────────────────┬───────────────────────┬─────────────────────────┬───────┬─────────────┬────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────────────┬─────────────┬─────────────┬──────────────┬──────────────┬───────────┬───────────┬────────────┬────────────┬───────────────────────┬───────────────────────┬──────────────────────┬───────────────┬─────────────┬────────────────────┬──────────────┬─────────┬──────────────────────────┬───────────┬────────────┬─────────────────────────┬─────────────────────────────────┐
│ cpu_usage_percent │ cpu_user_percent │ cpu_idle_percent │ cpu_system_percent │ cpu_wait_percent │ cpu_nice_percent │ cpu_interrupt_percent │ load_shortterm_percent │ load_midterm_percent │ load_longter

In [6]:
VM_HARDWARE = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/vms/2024-12-14T000000Z_2025-04-13T235959Z/vms.csv"
VM_DATA = "/Users/biancachiusano/Desktop/uva/thesis/Thesis/datasets/cloud_energy_consumption/nodes-vms/2024-12-14T000000Z_2025-04-13T235959Z/**/*.csv"

## VM HARDWARE

In [7]:
# VM HARDWARE 
con.query(f"""CREATE OR REPLACE TABLE vmhardware AS SELECT * FROM '{VM_HARDWARE}'""")
con.query("""SELECT * FROM vmhardware LIMIT 5""").show()

┌──────────┬──────────┬────────────┬───────────┬────────┬───────────┬─────────┬──────────────┐
│  vm_id   │ user_id  │ project_id │ image_ref │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │
│ varchar  │ varchar  │  varchar   │  varchar  │ double │  double   │ double  │    double    │
├──────────┼──────────┼────────────┼───────────┼────────┼───────────┼─────────┼──────────────┤
│ 6239bcf4 │ 39786247 │ 4ba6ce42   │ 85ecdff4  │   32.0 │   60000.0 │    30.0 │        160.0 │
│ 80458243 │ bd41a7e4 │ 74dfdf00   │ cb92696e  │    1.0 │    2000.0 │    10.0 │         20.0 │
│ 5d9f6237 │ 7190ddfa │ 906147a9   │ 78c5906a  │    4.0 │    7500.0 │    30.0 │         60.0 │
│ 941785dc │ 1a324c91 │ f09054da   │ 23658854  │    2.0 │    4000.0 │    10.0 │        100.0 │
│ fb856e0f │ c1eb1c1c │ f09054da   │ c015d680  │    4.0 │    7500.0 │    30.0 │         60.0 │
└──────────┴──────────┴────────────┴───────────┴────────┴───────────┴─────────┴──────────────┘



## VM DATA

In [8]:
con.execute(f"""
                CREATE OR REPLACE VIEW vm_data AS
                SELECT *
                FROM read_csv_auto('{VM_DATA}',
                                filename=false,
                                union_by_name=true)
            """)

### Merging

In [9]:
con.execute("""
    CREATE OR REPLACE TABLE vm_merged AS
    SELECT 
        d.*, 
        h.*
    FROM vm_data d
    LEFT JOIN vmhardware h
    ON d.vm_id = h.vm_id
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Imputing missing values
Forward Fill and 0s

In [10]:
con.execute("""
    CREATE OR REPLACE TABLE vm_filled AS
    SELECT *,
        LAST_VALUE(scaphandre_vm_power_total_watts IGNORE NULLS)
        OVER (
            PARTITION BY vm_id 
            ORDER BY timestamp
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS power_filled
    FROM vm_merged
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
con.execute("""
    CREATE OR REPLACE TABLE vm_final AS
    SELECT *,
        COALESCE(power_filled, 0) AS power_clean
    FROM vm_filled
 """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
con.query("""SELECT * FROM vm_final LIMIT 5""").show()

┌──────────────────────────┬──────────┬─────────────────┬──────────────────┬─────────────────────────────────┬──────────┬──────────┬────────────┬───────────┬────────┬───────────┬─────────┬──────────────┬──────────────┬─────────────┐
│        timestamp         │  vm_id   │ hypervisor_name │ hypervisor_group │ scaphandre_vm_power_total_watts │ vm_id_1  │ user_id  │ project_id │ image_ref │ vcpus  │ memory_mb │ root_gb │ ephemeral_gb │ power_filled │ power_clean │
│ timestamp with time zone │ varchar  │     varchar     │     varchar      │             double              │ varchar  │ varchar  │  varchar   │  varchar  │ double │  double   │ double  │    double    │    double    │   double    │
├──────────────────────────┼──────────┼─────────────────┼──────────────────┼─────────────────────────────────┼──────────┼──────────┼────────────┼───────────┼────────┼───────────┼─────────┼──────────────┼──────────────┼─────────────┤
│ 2025-02-06 18:27:00+01   │ c3cc7b6e │ e4362c19        │ a6177608  

In [13]:
#TIMESTAMP = '2025-01-16T08:33:00Z'
TIMESTAMP = '2025-01-16 08:33:00 UTC'

### Get VM list per node 

In [24]:
# Check timestamp notation
con.query(f"""
    SELECT 
        hypervisor_name AS node_name,
        LIST(vm_id) AS vm_ids
    FROM vm_final
    WHERE timestamp = '{TIMESTAMP}'
    GROUP BY hypervisor_name
""").show()

┌───────────┬──────────────────────────────────────────────────────────────────────────────────┐
│ node_name │                                      vm_ids                                      │
│  varchar  │                                    varchar[]                                     │
├───────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ 7c804642  │ [cb521e0f]                                                                       │
│ b654db47  │ [e80b91dd]                                                                       │
│ 0eb86326  │ [ef84013a]                                                                       │
│ c9900ab9  │ [5486492a, 654a92dd]                                                             │
│ 6bb5c698  │ [f545ef7a, 4aeb93ea, be1a8732, b963d466, eae4dca6]                               │
│ ea7f6999  │ [91dc05b7]                                                                       │
│ 6d834fb8  │ [c6b13b89]      

### VM Aggregation - get node load - use for comparison - ALSO nodes_t

In [16]:
con.query(f"""
    SELECT 
        hypervisor_name AS node_name,
        SUM(power_clean) AS total_power,
        COUNT(*) AS vm_count
    FROM vm_final
    WHERE timestamp = '{TIMESTAMP}'
    GROUP BY hypervisor_name
""").show()

┌───────────┬─────────────────────┬──────────┐
│ node_name │     total_power     │ vm_count │
│  varchar  │       double        │  int64   │
├───────────┼─────────────────────┼──────────┤
│ 82415d68  │                0.44 │       11 │
│ d8d6d1e0  │                0.11 │        1 │
│ 6342c79e  │                0.18 │        1 │
│ 7f4f3762  │                0.12 │        6 │
│ 28e8b849  │                0.01 │        1 │
│ 3d7960e7  │  1.1600000000000001 │        7 │
│ 58512373  │                0.25 │        7 │
│ 5c5e400b  │                0.07 │        1 │
│ 5333662e  │              118.55 │        1 │
│ e2b6ed0e  │   9.450000000000001 │        5 │
│    ·      │                  ·  │        · │
│    ·      │                  ·  │        · │
│    ·      │                  ·  │        · │
│ a22bdd7c  │                0.07 │        2 │
│ 7f384201  │ 0.22000000000000003 │        4 │
│ 6ff55332  │                0.24 │        3 │
│ 8d5a6c7e  │                0.41 │        1 │
│ 764b2e98  │

# Timestamps

In [17]:
timestamps = con.execute("""
    SELECT DISTINCT timestamp AT TIME ZONE 'UTC'
    FROM vm_final
    ORDER BY timestamp
""").fetchall()

In [14]:
LOWER_THRESHOLD = 10
UPPER_THRESHOLD = 85

In [45]:
def sortDecreasingUtilization(vmList, total_vm_power):

    # Input: List of vm_ids and corresponding power usage, and total vm power for the node
    # Output: List of vm_ids sorted by decreasing load (as a percentage of total vm power)
    vmList['vm_load'] = vmList['vm_power'] / total_vm_power * 100
    sorted_vm_df = vmList.sort_values(by='vm_load', ascending=False)
    return sorted_vm_df['vm_id'].to_list(), sorted_vm_df['vm_power'].to_list(), sorted_vm_df['vm_load'].to_list()

In [41]:
 # for each node at time t, extract name, cpu usage, and vm count and vm ids
node_df_t = con.execute("""
                        SELECT 
                            n.node_name,
                            n.cpu_usage_percent,
                            n.ipmi_system_power_watts, 
                            COUNT(v.vm_id) AS vm_count,
                            LIST(v.vm_id) AS vm_ids,
                            LIST(v.power_clean) AS vm_powers,
                            SUM(v.power_clean) AS total_vm_power
                        FROM nodes_table n
                        LEFT JOIN vm_final v
                            ON n.timestamp = v.timestamp
                            AND n.node_name = v.hypervisor_name
                        WHERE n.timestamp = ?
                        GROUP BY n.node_name, n.cpu_usage_percent, n.ipmi_system_power_watts
                    """, [TIMESTAMP]).df()

    
overloaded = node_df_t[node_df_t["cpu_usage_percent"] > UPPER_THRESHOLD]
underloaded = node_df_t[node_df_t["cpu_usage_percent"] < LOWER_THRESHOLD]


In [ ]:
overloaded['vm_loads'] = None
underloaded['vm_loads'] = None

for idx, row in overloaded.iterrows():
    # replace the original vm_ids and vm_powers with the sorted ones (and calculate vm load)
    vm_ids, vm_powers, vm_loads = sortDecreasingUtilization(
        pd.DataFrame({
            'vm_id': row['vm_ids'],
            'vm_power': row['vm_powers']
        }),
        row['total_vm_power']
    )

    overloaded.at[idx, 'vm_ids'] = vm_ids
    overloaded.at[idx, 'vm_powers'] = vm_powers
    overloaded.at[idx, 'vm_loads'] = vm_loads

for idx, row in underloaded.iterrows():
    # replace the original vm_ids and vm_powers with the sorted ones (and calculate vm load)
    vm_ids, vm_powers, vm_loads = sortDecreasingUtilization(
        pd.DataFrame({
            'vm_id': row['vm_ids'],
            'vm_power': row['vm_powers']
        }),
        row['total_vm_power']
    )

    underloaded.at[idx, 'vm_ids'] = vm_ids
    underloaded.at[idx, 'vm_powers'] = vm_powers
    underloaded.at[idx, 'vm_loads'] = vm_loads

In [43]:
overloaded

,node_name,cpu_usage_percent,ipmi_system_power_watts,vm_count,vm_ids,vm_powers,total_vm_power,vm_loads
27,05c5ef00,99.65,784.17,1,[06e83870],[521.42],521.42,[100.0]
32,46de937b,91.68,785.00,0,[None],[nan],NaN,[nan]
37,5433c3e6,99.95,780.83,1,[b3382983],[492.27],492.27,[100.0]
44,764b2e98,100.00,800.00,1,[525134f6],[525.23],525.23,[100.0]
59,2eea5215,85.99,785.00,0,[None],[nan],NaN,[nan]
74,88b02143,86.38,786.67,1,[857a76fa],[454.25],454.25,[100.0]
88,cbf4a974,99.92,788.33,1,[5d14c1f6],[524.4],524.40,[100.0]


In [44]:
underloaded

,node_name,cpu_usage_percent,ipmi_system_power_watts,vm_count,vm_ids,vm_powers,total_vm_power,vm_loads
1,5d7db037,0.16,415.00,7,"[95b66945, ef9152aa, 32d947ff, af20ec04, 0c898...","[0.1, 0.06, 0.03, 0.03, 0.02, 0.02, 0.0]",0.26,"[38.46153846153847, 23.076923076923077, 11.538..."
2,e41df102,0.32,416.67,5,"[ac0f1f33, fb856e0f, c9afd608, 6fcacc4e, 29919...","[0.43, 0.29, 0.03, 0.02, 0.01]",0.78,"[55.128205128205124, 37.179487179487175, 3.846..."
3,a639d055,0.81,278.33,1,[fe6a63da],[0.2],0.20,[100.0]
4,54f5286d,0.93,77.33,1,[465cef03],[0.5],0.50,[100.0]
6,a22bdd7c,0.07,407.50,2,"[b2f57d95, dd8ec43e]","[0.04, 0.03]",0.07,"[57.14285714285714, 42.85714285714285]"
...,...,...,...,...,...,...,...,...
122,d35334d1,0.34,283.33,1,[8a4874ee],[0.19],0.19,[100.0]
123,79b5307d,1.22,76.67,1,[15497bd1],[0.54],0.54,[100.0]
124,366801,0.62,412.50,0,[None],[nan],NaN,[nan]
125,a2ffa1e3,0.09,61.67,0,[None],[nan],NaN,[nan]


There are mony occurences in which the node is being underutilised but is not 0 and there are no VMs on them. 
This does not affect host detection but when it comes to vm selection there are no VMs to select and non to migrate so the node should be put to idle

In [ ]:
# Example


In [ ]:
def minimization_of_migrations(vms, loads, cpu_util, vms_to_migrate):
    best_fit_util = float('inf')
    while cpu_util > UPPER_THRESHOLD and vms:
        for i in range(len(vms)):
            if loads[i] > cpu_util - UPPER_THRESHOLD:
                util = loads[i] - cpu_util + UPPER_THRESHOLD
                print("NEW UTL:", util)
                if util < best_fit_util:
                    best_fit_util = util
                    best_fit_vm = vms[i]
        cpu_util -= best_fit_util
        print("CPU UTIL:", cpu_util)

        vms_to_migrate.append({
            "vm_id": best_fit_vm,
            "source_node": row["node_name"]
        })
        # remove the migrated vm from the list of vms and loads
        vms.remove(best_fit_vm)
        loads.remove(index=i)

In [ ]:
'''
    For underloaded nodes, we migrate all VMs to other nodes
    For overloaded nodes, we migrate VMs until we are under the upper threshold
'''
def selection_algorithm(overloaded, underloaded, t):
    vms_to_migrate = []

    # Underloaded nodes
    for _, row in underloaded.iterrows():
        if row["vm_count"] == 0:
            # TODO: should I put these nodes to sleep?
            continue
        
        for vm_id in row["vm_ids"]:
            vms_to_migrate.append({
            "vm_id": vm_id,
            "source_node": row["node_name"]
            })

    # Overloaded nodes
    for _, row in overloaded.iterrows():
        if row["vm_count"] == 0:
            continue
        
        vm_ids = row['vm_ids']
        vm_loads = row['vm_loads']
        host_util = row["cpu_usage_percent"]
        # Minimization of migrations policy

        if len(vm_ids) > 1:
            vms_to_migrate = minimization_of_migrations(vm_ids, vm_loads, host_util, vms_to_migrate)
        else:
            vms_to_migrate.append({
                "vm_id": vm_ids[0],
                "source_node": row["node_name"]
            })

    return vms_to_migrate

In [ ]:
for (t,) in timestamps: 
    
    # for each node at time t, extract name, cpu usage, and vm count and vm ids
    node_df_t = con.execute("""
                            SELECT 
                                n.node_name,
                                n.cpu_usage_percent,
                                COUNT(v.vm_id) AS vm_count,
                                LIST(v.vm_id) AS vm_ids
                            FROM nodes_table n
                            LEFT JOIN vm_final v
                                ON n.timestamp = v.timestamp
                                AND n.node_name = v.hypervisor_name
                            WHERE n.timestamp = ?
                            GROUP BY n.node_name, n.cpu_usage_percent
                        """, [t + " UTC"]).df()
    
    # Host Detection
    overloaded = node_df_t[node_df_t["cpu_usage_percent"] > UPPER_THRESHOLD]
    underloaded = node_df_t[node_df_t["cpu_usage_percent"] < LOWER_THRESHOLD]

    # VM Selection

    

In [ ]:
# Check if nodes with no vms have cpu usage to 0 -> they don't

In [ ]:
# total power consumption of the cluster at time t
